# Convolutional Neural Network

**The big picture:** the data is a folder of **photos of cats and dogs**. The goal is to build a model that looks at a new photo and says **"cat" or "dog"**.

A **convolutional neural network (CNN)** is a neural network built for images. It doesn't look at every pixel separately. Instead it slides small "detectors" across the picture, the way your eye scans it. Early detectors spot simple things like edges and corners. Later ones combine those into ears, eyes and fur patterns. At the end, an ordinary neural network uses those features to decide: cat or dog.

**The images are not included in this repo.** The code expects this folder layout next to the notebook:

```
dataset/
  training_set/
    cats/   (cat photos)
    dogs/   (dog photos)
  test_set/
    cats/
    dogs/
  single_prediction/
    cat_or_dog_1.jpg
```

The **subfolder names are the labels**. Keras learns that everything in `cats/` is a cat and everything in `dogs/` is a dog. Nobody has to label each photo by hand.

### Before you start
- **Read first:** **artificial_neural_network**, because a CNN ends with the same Dense layers, sigmoid output, adam optimiser and binary cross-entropy loss; this notebook only adds the image-specific front part.
- **New words in this notebook:** *convolution*, *filter (feature detector)*, *max pooling*, *flattening*, *image augmentation*, *overfitting*. Look them up in the [glossary](../../../GLOSSARY.md) if you get stuck.

### Importing the libraries

- **tensorflow** is Google's deep-learning library. Its **Keras** part (`tf.keras`) gives ready-made building blocks for neural networks, and also tools for reading images from folders.
- `tf.__version__` just shows which version is installed. This notebook is written for TensorFlow 2.16 or newer (Keras 3).

In [ ]:
import tensorflow as tf

In [0]:
tf.__version__

## Part 1 - Data Preprocessing

Before training, the photos must be read in, shrunk to one standard size, and turned into numbers. Each pixel becomes three numbers: how much **red, green and blue** it contains.

### Preprocessing the Training set

`image_dataset_from_directory` reads the photos from `dataset/training_set`:
- `image_size = (64, 64)` shrinks every photo to **64 × 64 pixels**. That's small, but enough to tell cats from dogs, and it keeps training fast.
- `batch_size = 32` sends the photos to the network **32 at a time**.
- `label_mode = 'binary'` means there are **two classes** (cat / dog), so each label is a single 0 or 1.
- `seed = 42` fixes the random order the photos are shuffled in, so every run is the same.
- `class_names` is the list of **folder names in alphabetical order**: `['cats', 'dogs']`. Their position is their label, so **cats = 0 and dogs = 1**.

The photos come out with pixel values from 0 to 255. Scaling them down and adding random changes (*augmentation*) happens inside the network itself; see "Initialising the CNN" below.

In [ ]:
training_set = tf.keras.utils.image_dataset_from_directory('dataset/training_set',
                                                           image_size = (64, 64),
                                                           batch_size = 32,
                                                           label_mode = 'binary',
                                                           seed = 42)
class_names = training_set.class_names
print(class_names)

### Preprocessing the Test set

- The test photos are read the same way: **64 × 64**, batches of **32**, binary labels.
- `shuffle = False` keeps them in a fixed order. There is no need to shuffle photos that are only used for checking.

**What about augmentation?** The random changes are layers inside the network, and Keras only switches them on while it is *learning*. When the network is checked on the test set or asked about a new photo, those layers do nothing. The test set plays the role of real, new photos, so we want to measure the network on them **as they are**.

In [ ]:
test_set = tf.keras.utils.image_dataset_from_directory('dataset/test_set',
                                                       image_size = (64, 64),
                                                       batch_size = 32,
                                                       label_mode = 'binary',
                                                       shuffle = False)

## Part 2 - Building the CNN

The network is built layer by layer:

**photo (64×64×3) → convolution → pooling → convolution → pooling → flatten → 128 neurons → 1 output (cat or dog)**

The first part (convolution and pooling) **finds features** in the image. The last part (the dense layers) **makes the decision**.

![CNN pipeline: 64x64x3 photo, convolution, pooling, convolution, pooling, flatten to 6,272 numbers, 128 neurons, 1 output](../../../docs/diagrams/cnn_pipeline.svg)

*Convolution and pooling shrink the photo while finding features, then flatten and the dense layers decide: cat or dog.*

### Initialising the CNN

- `Sequential()` creates an **empty network** whose layers will run one after another in a line.
- `Input(shape = (64, 64, 3))` says what goes in: photos of **64 × 64 pixels with 3 colour channels** (red, green, blue).
- `Rescaling(1./255)`: pixel values go from 0–255 down to **0–1**. This is the image version of feature scaling, and it keeps the maths stable. Because it is part of the network, any photo you give it later is scaled automatically.

The next three layers do **image augmentation**: they make random small changes to each training photo every time the network sees it, so it never sees exactly the same picture twice.
- `RandomShear(0.2)`: randomly **slants** the image a little, like pushing the top of a picture sideways.
- `RandomZoom(0.2)`: randomly **zooms in or out** by up to 20%.
- `RandomFlip('horizontal')`: randomly **mirrors** the image left-to-right. A dog facing left is still a dog.

**Why?** Without this, the network might memorise the exact training photos, for example "this particular dog in this exact position", and do badly on new ones. This is called **overfitting**. Augmentation teaches it that a cat is still a cat when it is tilted, closer, or facing the other way. These layers are only active during training.

In [ ]:
cnn = tf.keras.models.Sequential()
cnn.add(tf.keras.Input(shape = (64, 64, 3)))
cnn.add(tf.keras.layers.Rescaling(1./255))
cnn.add(tf.keras.layers.RandomShear(x_factor = 0.2, y_factor = 0.2))
cnn.add(tf.keras.layers.RandomZoom(0.2))
cnn.add(tf.keras.layers.RandomFlip('horizontal'))

### Step 1 - Convolution

**Convolution** slides a small window over the picture and checks for a pattern at each spot. It's like running a small stencil over the photo and noting where it matches.

- `filters=32`: the layer learns **32 different detectors** (also called *filters*). Each one looks for its own pattern, such as a vertical edge, a curve or a patch of a certain colour. Nobody tells them what to look for; they learn it during training.
- `kernel_size=3`: each detector is a **3 × 3 pixel** window.
- `activation='relu'`: negative results become 0, positive results pass through. In plain words, it keeps only the places where a pattern was found.
- The layer knows its input is a 64 × 64 × 3 photo from the `Input` line above.

The output is 32 "feature maps", one per detector, each showing **where** in the image its pattern appeared. Each map is 62 × 62, slightly smaller than the photo because the 3 × 3 window can't hang off the edge.

In [0]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))

### Step 2 - Pooling

**Max pooling** shrinks each feature map by keeping only the **strongest signal** in each small area.

- `pool_size=2`: looks at **2 × 2** squares of the feature map...
- `strides=2`: ...moving **2 pixels** at a time, so the squares don't overlap...
- ...and keeps only the **largest** of the 4 numbers in each square.

This halves the width and height (62 × 62 becomes 31 × 31).

**Why?** It makes the network faster (fewer numbers to handle). It also makes it care less about **exactly** where a feature is. An ear shifted a few pixels to the left still counts as an ear.

In [0]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

- Another **convolution** (32 detectors of 3 × 3, relu) followed by another **max pooling** (2 × 2).
- No `input_shape` this time: Keras knows the input is the output of the previous layer.

**Why a second round?** The first layer finds simple things like edges. This one works on *those* results, so it can combine edges into bigger shapes, such as an eye, an ear or a snout. The feature maps shrink again: 31 × 31 becomes 29 × 29 after convolution, then **14 × 14** after pooling.

In [0]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

- `Flatten()` takes the stack of feature maps (14 × 14 × 32) and lays all the numbers out **in one long line**: 14 × 14 × 32 = **6,272 numbers**.

**Why?** The decision-making layers that come next expect a simple list of inputs, like the columns of a table, not a 3D block. Each number in the list now means something like "how strongly feature X appeared in area Y".

In [0]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

- A normal **fully connected** (`Dense`) layer, the same kind used in the ANN notebook. Every one of its neurons receives all 6,272 numbers.
- `units=128`: **128 neurons**. This is more than the ANN's 6, because there is much more information to combine here.
- `activation='relu'`, as before.

This is where the network **reasons about** the features it found. For example: "pointy ears + whiskers + small snout usually means cat".

In [0]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5 - Output Layer

- `units=1`: **one neuron**, because there is one yes/no question: is it a dog?
- `activation='sigmoid'` squashes the result into a number **between 0 and 1**. Close to **1 means dog** and close to **0 means cat** (see the single prediction below for why).

In [0]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

Training means showing the network all the training photos many times and nudging its weights after each batch so its guesses improve. This includes the weights inside the 32 + 32 detectors, so the network learns **what patterns to look for** as well as how to use them.

### Compiling the CNN

Exactly the same settings as the ANN, because this is also a two-answer (binary) problem:

- `optimizer = 'adam'`: the method for nudging the weights. It takes small steps in whichever direction makes the error smaller, and adjusts its step size automatically.
- `loss = 'binary_crossentropy'`: the way of **measuring how wrong** a guess is when there are two possible answers. Confident wrong answers cost the most.
- `metrics = ['accuracy']`: also report the **percentage of correct guesses**, so we can watch progress.

In [0]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

- `x = training_set`: learn from the training photos, 32 at a time (the augmentation layers change them a little each time).
- `validation_data = test_set`: after each epoch, also check the network on the test photos. These are **only used for checking, never for learning**.
- `epochs = 25`: go through the whole training set **25 times**. That is fewer than the ANN's 100, because each pass over thousands of images takes much longer.

**How to read the output** (not saved in this notebook; you'll see it when you run it): each epoch prints a line with four numbers.
- `loss` and `accuracy` are for the **training** photos.
- `val_loss` and `val_accuracy` are for the **test** photos.
- Both accuracies should climb over the epochs. If training accuracy keeps rising but `val_accuracy` stalls far below it, the network is starting to memorise the training photos (overfitting).

In [0]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

## Part 4 - Making a single prediction

Now we give the trained network **one new photo** and ask: cat or dog?

- `load_img(..., target_size = (64, 64))` opens the photo and shrinks it to **64 × 64**, the same size the network was trained on.
- `img_to_array` turns the picture into a block of numbers: 64 × 64 × 3.
- `np.expand_dims(..., axis = 0)` wraps it in an extra layer to make a **batch of one**: 1 × 64 × 64 × 3. **Why?** The network was trained on batches, so `predict` always expects a batch, even if it only holds one image.
- There is no need to divide by 255 here: the network's own `Rescaling` layer does it, and the random augmentation layers switch off automatically when predicting.
- `cnn.predict(test_image)` returns the sigmoid output, a number between 0 and 1. `result[0][0]` picks it out (first image in the batch, the only output neuron).
- `print(class_names)` shows `['cats', 'dogs']`: position 0 is cats and position 1 is dogs. That's why **near 1 means dog** and **near 0 means cat**.
- The `if` turns the number into the word `'dog'` or `'cat'`: above 0.5 means the network leans towards dog, and `print(prediction)` shows it.

**In one sentence:** read cat and dog photos from labelled folders, build a network that scales each photo, adds random tilts, zooms and flips while training, scans for visual patterns (convolution), shrinks them (pooling), lines them up (flatten) and decides with ordinary neurons, train it for 25 passes, then ask it about one new photo.

In [ ]:
import numpy as np
test_image = tf.keras.utils.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = tf.keras.utils.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
print(class_names)
if result[0][0] > 0.5:
  prediction = 'dog'
else:
  prediction = 'cat'

In [0]:
print(prediction)

## Common mistakes to avoid
- **Scaling the new photo twice.** The network already contains a `Rescaling(1./255)` layer, so the single photo must go in with its raw 0–255 pixels. If you also divide by 255 yourself, the network sees values close to 0 and its answer is unreliable.
- **Mixing up which number means cat and which means dog.** Keras numbers the classes from the folder names in alphabetical order, so `['cats', 'dogs']` means cats = 0 and dogs = 1. Print `class_names` to check before reading the result, especially if you rename the folders.
- **Putting augmentation outside the network and applying it to the test set.** Random shear, zoom and flip are a training trick. Keeping them as layers inside the model is safe, because Keras switches them off automatically when evaluating and predicting.
- **Changing the image size in only one place.** `image_size = (64, 64)` appears in both dataset cells, `target_size = (64, 64)` in `load_img`, and `shape = (64, 64, 3)` in the `Input` line. If they don't all match, you get a shape error.

## Try this yourself
Change one thing, rerun the notebook, and see what happens.

1. You need the `dataset/` folder of cat and dog photos (see the first explanation cell) for all of these. In the training cell, change `epochs = 25` to `epochs = 5` (rebuild from `cnn = tf.keras.models.Sequential()` first). What to notice: compare `accuracy` and `val_accuracy` on the last line with a 25-epoch run. Fewer passes should leave both lower.
2. Turn augmentation off: in the cell that starts `cnn = tf.keras.models.Sequential()`, delete the `RandomShear`, `RandomZoom` and `RandomFlip` lines, then run the build cells and retrain. What to notice: training `accuracy` will likely climb higher, but watch whether `val_accuracy` falls further behind it. That growing gap is overfitting.
3. Add a third convolution and pooling round: after the second-layer cell, add a cell with `cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))` and `cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))`, then run the remaining build cells and retrain. What to notice: whether `val_accuracy` changes. The feature maps shrink to 6 × 6, so Flatten now gives far fewer numbers than 6,272.
4. In the single-prediction cell, add `print(result[0][0])` after `result = cnn.predict(test_image)`, and try a few of your own photos by changing the file path. What to notice: values near 0 or 1 mean the network is confident; values near 0.5 mean it is unsure.

## Check your understanding
Try to answer in your head first, then click to reveal.

**1. The random shear, zoom and flip are layers inside the network. Why don't they spoil the test-set score or the single prediction?**

<details><summary>Show answer</summary>

Keras only switches those random layers on while the network is learning. When it is evaluated on the test set or asked to predict, they pass the photo through unchanged, so the network is judged on photos as they really are.

</details>

**2. What does max pooling do to a 62 × 62 feature map here, and why is that useful?**

<details><summary>Show answer</summary>

With `pool_size=2` and `strides=2` it keeps only the biggest number in each 2 × 2 square, shrinking the map to 31 × 31. That means fewer numbers to process, and the network cares less about the exact position of a feature.

</details>

**3. If `result[0][0]` is 0.92, what does the notebook print, and why?**

<details><summary>Show answer</summary>

It prints `dog`. 0.92 is above 0.5, and the folders are numbered alphabetically (`['cats', 'dogs']`, so cats = 0 and dogs = 1), so values near 1 mean dog.

</details>

**4. What would happen if training `accuracy` reached 98% but `val_accuracy` stayed around 75%?**

<details><summary>Show answer</summary>

The network would be overfitting: memorising the training photos rather than learning what cats and dogs look like in general. More augmentation, more training photos or fewer epochs are common ways to narrow the gap.

</details>

---
📚 [Learning path](../../../LEARNING_PATH.md) · [Glossary](../../../GLOSSARY.md)